In [ ]:
!pip install onnx onnxruntime onnx-tool numpy pandas -q

In [ ]:
import os, sys, json, zipfile, math, re
from pathlib import Path
import numpy as np
import onnx
import onnxruntime as ort
print('ENSEMBLE v139b - Added Afr1ste 6225')

sys.path.insert(0, '/kaggle/input/competitions/neurogolf-2026/neurogolf_utils')
import neurogolf_utils as nu
ort.set_default_logger_severity(3)

OUTPUT_DIR = Path('/kaggle/working')
SUB_DIR = OUTPUT_DIR / 'submission'
SUB_DIR.mkdir(parents=True, exist_ok=True)
print('Config done')


In [ ]:
# NOTEBOOK SOURCES (highest priority)
NOTEBOOK_SOURCES = [
    ('NGC_Mix', Path('/kaggle/input/notebooks/jonathanchan/ngc26-constraint-smart-logic-mix-blending/submission.zip')),
    ('Konbu_341', Path('/kaggle/input/notebooks/konbu17/neurogolf-2026-blended-341-tasks-lb-4215/submission.zip')),
    ('Magma_4200', Path('/kaggle/input/notebooks/magmacot/4200-v5-neurogolf-fix-for-new-system-soon/submission.zip')),
    ('Afr1ste_6225', Path('/kaggle/input/notebooks/afr1ste/neurogolf-6225-51-public-score-open-solution/submission.zip')),
]

# ALL DATASETS
ENSEMBLE_SOURCES = [
    # Top scorers
    ('Secret', Path('/kaggle/input/datasets/svanikkolli/secret-dataset')),
    ('5550', Path('/kaggle/input/datasets/svanikkolli/5550-dataset')),
    ('LB4995', Path('/kaggle/input/datasets/svanikkolli/lb-4995')),
    ('NGC26', Path('/kaggle/input/datasets/svanikkolli/ngc26-dataset')),
    # Others
    ('345ONNX', Path('/kaggle/input/datasets/svanikkolli/345-onnx-submission-dataset')),
    ('ArcDataset', Path('/kaggle/input/datasets/svanikkolli/arc-dataset')),
    ('Artem', Path('/kaggle/input/datasets/svanikkolli/artem-nazemtsev-datasets')),
    ('Boost', Path('/kaggle/input/datasets/svanikkolli/boost-dataset')),
    ('Constraint', Path('/kaggle/input/datasets/svanikkolli/constraint-smart-logic-ensemble')),
    ('Cross', Path('/kaggle/input/datasets/svanikkolli/cross-dataset')),
    ('Geometric', Path('/kaggle/input/datasets/svanikkolli/geometric-dataset')),
    ('Agent', Path('/kaggle/input/datasets/sigmaborov/golf-solve-agent')),
    ('Infinitesimals', Path('/kaggle/input/datasets/svanikkolli/infinitesimals-dataset')),
    ('Jiwei_liu', Path('/kaggle/input/datasets/svanikkolli/jiwei-liu-dataset')),
    ('Logic', Path('/kaggle/input/datasets/svanikkolli/logic-dataset')),
    ('Magmucot', Path('/kaggle/input/datasets/svanikkolli/magmucot-dataset')),
    ('MEFA', Path('/kaggle/input/datasets/svanikkolli/mefa-agi-ensemble-dataset')),
    ('Starter', Path('/kaggle/input/datasets/svanikkolli/neurogolf-2026-starter')),
    ('YashNB', Path('/kaggle/input/datasets/yash9439/neurogolf-submission-v1')),
    ('Konbu_Blend', Path('/kaggle/input/datasets/konbu17/neurogolf-2026-blended-341')),
    ('NNMax', Path('/kaggle/input/datasets/svanikkolli/nnmax-dataset')),
    ('RuleBased', Path('/kaggle/input/datasets/svanikkolli/rule-based-dataset')),
    ('Saxaphone', Path('/kaggle/input/datasets/svanikkolli/saxaphone-dataset')),
    ('Scorer', Path('/kaggle/input/datasets/svanikkolli/scorer-arc-appx')),
    ('TinyONNX', Path('/kaggle/input/datasets/svanikkolli/tiny-onnx-dataset')),
    ('Triage', Path('/kaggle/input/datasets/svanikkolli/triage-system-dataset')),
    ('Watermelon2', Path('/kaggle/input/datasets/svanikkolli/watermelon-2-dataset')),
    ('Watermelon', Path('/kaggle/input/datasets/svanikkolli/watermelon-dataset')),
    ('Yash', Path('/kaggle/input/datasets/svanikkolli/yash-dataset')),
    ('Yash2', Path('/kaggle/input/datasets/svanikkolli/yash-dataset')),
    ('Karnakbaev', Path('/kaggle/input/datasets/karnakbaevarthur/logic-for-each-arc-task')),
    ('Konbu_401', Path('/kaggle/input/datasets/konbu17/neurogolf-2026-blended-401-v117')),
]


In [ ]:
def load_zip(zip_path, label):
    models = {}
    if not zip_path.exists(): return models
    try:
        with zipfile.ZipFile(zip_path, 'r') as zf:
            for entry in zf.namelist():
                m = re.match(r'task(\d{3})\.onnx', os.path.basename(entry))
                if m: models[int(m.group(1))] = zf.read(entry)
        if models: print(f'  [{label}] {len(models)}')
    except: pass
    return models

def load_dataset(path, label):
    models = {}
    if not path.exists(): return models
    try:
        for f in path.glob('*.zip'):
            with zipfile.ZipFile(f, 'r') as z:
                for name in z.namelist():
                    m = re.match(r'task(\d{3})\.onnx', os.path.basename(name))
                    if m: models[int(m.group(1))] = z.read(name)
            if models: print(f'  [{label}] {len(models)}'); return models
        for f in path.glob('*.onnx'):
            m = re.match(r'task(\d{3})\.onnx', f.name)
            if m: models[int(m.group(1))] = f.read_bytes()
        if models: print(f'  [{label}] {len(models)}')
    except: pass
    return models

print('Loading EVERYTHING...')
all_sources = {}

# Notebooks first
for label, path in NOTEBOOK_SOURCES:
    models = load_zip(path, label)
    if models: all_sources[label] = models

# Then all datasets
for label, path in ENSEMBLE_SOURCES:
    models = load_dataset(path, label)
    if models: all_sources[label] = models

print(f'TOTAL: {sum(len(m) for m in all_sources.values())} models from {len(all_sources)} sources')


In [ ]:
def validate_official(task_id, raw):
    try:
        sess = ort.InferenceSession(raw, providers=['CPUExecutionProvider'])
    except: return None
    try:
        examples = nu.load_examples(task_id)
    except: return None
    try:
        agi_pass, agi_fail, _ = nu.verify_subset(sess, examples['train'] + examples['test'])
        gen_pass, gen_fail, _ = nu.verify_subset(sess, examples['arc-gen'])
    except: return None
    if agi_fail > 0 or gen_fail > 0: return None
    try:
        tmp = f'/tmp/v{task_id:03d}.onnx'
        with open(tmp, 'wb') as f: f.write(raw)
        macs, mem, params = nu.score_network(tmp)
        if None in (macs, mem, params): return None
        cost = macs + mem + params
    except: return None
    return {'cost': cost, 'score': max(1.0, 25.0 - math.log(cost))}


In [ ]:
print('=== ENSEMBLE v139 - EVERYTHING ===')

solved = {}
results = []

for task_id in range(1, 401):
    candidates = []
    for src in all_sources:
        if task_id not in all_sources[src]: continue
        raw = all_sources[src][task_id]
        r = validate_official(task_id, raw)
        if r: candidates.append((r['cost'], r['score'], raw, src))
    
    if candidates:
        candidates.sort(key=lambda x: (x[0], -x[1]))
        cost, score, raw, src = candidates[0]
        solved[task_id] = raw
        results.append({'task': task_id, 'cost': cost, 'score': score, 'source': src})
    
    if task_id % 50 == 0: print(f'  [{task_id}] solved={len(solved)}')

print(f'\nSolved: {len(solved)}')


In [ ]:
import pandas as pd
df = pd.DataFrame(results)
print('=== Results ===')
print(f'Solved: {len(solved)}/400')
print(f'Score: {df["score"].sum():.2f}')
print()
print(df['source'].value_counts())


In [ ]:
print('=== Saving ===')
for tid, raw in solved.items():
    (SUB_DIR / f'task{tid:03d}.onnx').write_bytes(raw)
zip_path = OUTPUT_DIR / 'submission.zip'
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(SUB_DIR.glob('*.onnx')): zf.write(f, f.name)
print(f'Wrote {len(solved)} files')
